# AIMENG：Google Colab 免费 GPU 训练入口

本 notebook 面向免费 Colab GPU。它先检查 GPU、磁盘与内存，再获取固定的 Ornith GGUF 并生成 manifest。**GGUF 本身作为冻结推理基线，不进行假装可行的常规反向传播。**

真正训练路由器需要来自实际后端的测量轨迹：每个样本至少包含可观测特征、候选通道组/候选池、质量分数、延迟和峰值内存。没有真实轨迹时，只运行环境检查，不会伪造训练数据或宣称训练成功。

In [ ]:
import os, sys, shutil, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('RAM (GiB):', round(__import__('psutil').virtual_memory().total / 2**30, 2))
print('Free disk (GiB):', round(shutil.disk_usage('/content').free / 2**30, 2))
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('GPU VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
except Exception as e:
    print('PyTorch/GPU probe failed:', repr(e))
assert shutil.disk_usage('/content').free > 8 * 2**30, 'Colab 可用磁盘不足 8 GiB；不要开始下载。'


## 1. 获取仓库代码
在 Colab 菜单选择 **运行时 → 更改运行时类型 → T4 GPU（如果账号当时提供）**。免费 GPU 的可用性、型号和时长由 Google 动态分配，并不保证每次都有。

In [ ]:
import os, subprocess
REPO = 'https://github.com/bslsjdk/aimeng.git'
BRANCH = 'feat/gguf-router-training-foundation'
if not os.path.isdir('/content/aimeng/.git'):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,'/content/aimeng'], check=True)
else:
    subprocess.run(['git','-C','/content/aimeng','fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C','/content/aimeng','checkout',BRANCH], check=True)
%cd /content/aimeng
!python -m pip install -q -r requirements.txt psutil

## 2. 下载并核对 GGUF
文件约 5.8 GB。Colab 免费实例的磁盘空间会变化；若空间不足，停止而不是反复下载。模型不会上传到 Git。

In [ ]:
from pathlib import Path
free_gib = shutil.disk_usage('/content').free / 2**30
print(f'当前剩余磁盘：{free_gib:.2f} GiB')
if free_gib < 8:
    raise RuntimeError('磁盘空间不足 8 GiB，已停止，避免下载中断。')
subprocess.run([sys.executable, 'scripts/fetch_model.py'], check=True)
subprocess.run([sys.executable, 'scripts/inspect_gguf.py', '--output', 'runs/gguf_manifest.json'], check=True)
print(Path('runs/gguf_manifest.json').read_text()[:6000])

## 3. 真实路由训练前置检查
当前仓库的训练目标是独立的小型路由器，不是更新 9B GGUF 权重。训练轨迹必须由具备内部激活观测和真实通道跳过能力的推理后端产生。只有普通问答文本或模型自评，不能证明哪些 FFN 通道可以安全跳过，也不能证明节省了计算/内存。

当真实后端输出 `data/router_traces.csv` 后，才应在 GPU 上训练路由器。至少需要 `feature_*` 特征列以及 `reward` 标签列；reward 必须由实测任务质量、延迟、峰值内存等指标计算，不能用随机数替代。

In [ ]:
from pathlib import Path
import pandas as pd
trace = Path('data/router_traces.csv')
if not trace.exists():
    print('STOP: 未找到 data/router_traces.csv。当前仓库还没有真实后端轨迹，因此不能诚实地训练有效的通道路由器。')
    print('本次完成的只是 GPU 环境检查、GGUF 下载与元数据核验，不会生成伪造数据。')
else:
    df = pd.read_csv(trace)
    feature_cols = [c for c in df.columns if c.startswith('feature_')]
    if not feature_cols or 'reward' not in df.columns:
        raise ValueError('轨迹格式不正确：需要 feature_* 列和 reward 列。')
    if len(df) < 100:
        raise ValueError(f'真实轨迹只有 {len(df)} 行，至少 100 行才允许进行初步实验。')
    if df[feature_cols + ['reward']].isna().any().any():
        raise ValueError('训练轨迹包含缺失值，请先修复数据。')
    print('轨迹检查通过。样本数:', len(df), '特征数:', len(feature_cols))
    print('注意：格式检查通过不等于标签有效；需确认 reward 来自真实后端测量。')

## 4. 下一阶段的验收标准
- 保存训练配置、随机种子、轨迹数据版本和 checkpoint。
- 使用留出任务比较全量路径与路由路径的质量、首 token 延迟、tokens/s、峰值 RSS/PSS 和计算量。
- 质量门槛不通过时必须回退全量路径。
- Android 端单独测整应用峰值 RAM，硬限制仍是 4096 MiB，不能用 Colab 显存结果替代。

不要把这份 notebook 的环境检查结果写成“路由器训练成功”。